# 1D Tune and Relaunch a Deployment

**Purpose:**
Change how vLLM serves your model, by editing the **vLLM Arguments** and relaunching, and confirm each change from the client side.

**What we're doing**

We read the limits the running instance has now, export its configuration, relaunch it with a shorter context length and a cap on concurrent sequences, then test both limits.

**Why we're doing it**

* The recipe gets a model running. Sizing it for your users and your GPU is a separate decision, and vLLM arguments are where you make it
* vLLM reads its arguments only at launch, so every change is a relaunch. Export and Import Configuration make that quick and repeatable
* Knowing what each limit looks like from a client, an error or a queue, makes it easier to diagnose later

**What to look for**

* `max_model_len` changing in the `/models` response
* A request that is too long rejected with a clear error instead of failing silently
* Requests above the concurrency cap waiting in a queue instead of failing

**Output**

A relaunched instance with your arguments, and `.env` pointing at it.

> **Prerequisite:** Section 1 is complete, and ideally 1B, whose concurrency test this notebook reuses. Relaunching gives you a new instance: if you have already set up Section 2, read 1D.8 before you start.

> **Note:** This notebook stops the original instance and starts another, so the GPU loads the model again. Allow the same few minutes the first launch took.


## 1D.1 Load Values from `.env`

Run the cell below. It reads the `.env` file you created in Section 1.6, so the endpoint, model name, and token are the same ones you used there.

In [ ]:
import sys
sys.path.insert(0, "..")
from config import KLIQUE_BASE_URL as base_url, KLIQUE_MODEL as model, KLIQUE_GATEWAY_TOKEN as token
import requests

url_models = f"{base_url}/models"
url_chat = f"{base_url}/chat/completions"
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}

print(f"Endpoint: {base_url}")
print(f"Model:    {model}")
print(f"Token:    {token[:8]}...")

import json
import time
from concurrent.futures import ThreadPoolExecutor

## 1D.2 Read the Current Limits

vLLM's `/models` response includes more than the name. Two fields matter here:

| Field | Meaning |
|---|---|
| `root` | The model the server actually loaded, such as `Qwen/Qwen3.5-2B` |
| `max_model_len` | The longest sequence the server accepts, in tokens: the prompt plus the reply |

The recipe does not set `--max-model-len`, so vLLM uses the longest context the model supports.

In [ ]:
def show_models(url):
    """Print each served model's name, loaded model, and context length. Returns the list, or None."""
    response = requests.get(f"{url}/models", headers=headers, timeout=60)
    if not response.ok:
        print(f"❌ HTTP {response.status_code}: {response.text[:300]}")
        return None
    served = response.json()["data"]
    for m in served:
        print(f"id: {m['id']}\n\troot: {m.get('root')}\n\tmax_model_len: {m.get('max_model_len')}")
    return served


before = show_models(base_url)

Write down `max_model_len`. A long context is useful, but vLLM reserves GPU memory for the **KV cache**, the per-token state of every active request. The longer each request may be, the fewer requests fit in that memory at once.

Open the instance's **console log** on its dashboard and look for the lines vLLM prints after loading, similar to:

```
GPU KV cache size: <N> tokens
Maximum concurrency for <max_model_len> tokens per request: <X>x
```

The second line is how many requests of the maximum length fit in the KV cache at the same time.

<!-- screenshot: images/console_kv_cache.png (console log with the KV cache size and maximum concurrency lines) -->

## 1D.3 Choose the New Arguments

We add two arguments to the end of the recipe's arguments:

| Argument | Value | Effect |
|---|---|---|
| `--max-model-len` | `4096` | Rejects any request whose prompt plus `max_tokens` exceeds 4,096 tokens. Shorter requests leave room for more of them in the KV cache. |
| `--max-num-seqs` | `4` | Runs at most 4 requests at a time. Additional requests wait in vLLM's queue until a slot frees. |

`4` is deliberately low so the effect is easy to see in 1D.6. A real deployment uses a higher value, chosen from the measurements in Section 1B.

Other arguments you will see in recipes and the vLLM documentation:

| Argument | What it controls |
|---|---|
| `--gpu-memory-utilization` | The fraction of GPU memory vLLM may use, for the weights plus the KV cache. vLLM's default is `0.9`. Lower it when something else must share the GPU. |
| `--served-model-name` | The name clients send as `"model"`, instead of the model's full name. Changing it changes your **Model Endpoint Name**; see 1D.8. |
| `--tensor-parallel-size` | How many GPUs one model is split across. The recipe sets `1`. |

## 1D.4 Relaunch with the New Arguments

1. In the instance list, open the `qwen-lab` instance's menu (⋮) and choose **Export Configuration**. Keep the downloaded file.
2. In the same menu, choose **Stop**. This frees the GPU for the new instance. On a queue with a single GPU, the new instance would otherwise wait in **Loading**.
3. In the instance list menu, choose **Import Configuration** and select the file. The launch form opens with your Section 1.3 values.
4. Change **Instance Name** to `qwen-lab-tuned`.
5. Under *Deployed Models*, click **EDIT** and add the new arguments to the end of **vLLM Arguments**:

   ```
   Qwen/Qwen3.5-2B --trust-remote-code --tensor-parallel-size 1 --enable-auto-tool-choice --tool-call-parser qwen3_coder --reasoning-parser qwen3 --max-model-len 4096 --max-num-seqs 4
   ```

6. Check that **Environment Variables** still has `VLLM_USE_RUST_FRONTEND` = `1`, and that **Advanced → Idle Time Limit (Hours)** is set. Click **Launch**.

<!-- screenshot: images/import_tuned_args.png (launch form from Import Configuration, with the extra vLLM Arguments) -->

Watch the new instance come up as in Section 1.4. In its console log, the **Maximum concurrency** line now reports a much higher multiple, for requests of 4,096 tokens.

> **Checkpoint:** `qwen-lab-tuned` is **Active** and appears in **Model Endpoints → Active**. Copy its endpoint URL.

## 1D.5 Point the Notebook at the New Endpoint

The new instance gets its own endpoint. Run the cell and paste the URL from Model Endpoints. Press Enter without typing anything to keep the URL from `.env`, for example if it did not change. The cell only tests the URL; 1D.8 updates `.env`.

In [ ]:
new_url = input("New endpoint URL (Enter to keep .env): ").strip().rstrip("/")
if not new_url:
    new_url = base_url
elif not new_url.endswith("/v1"):
    new_url += "/v1"

def context_length(served):
    """Return max_model_len for the entry named KLIQUE_MODEL, or None if it is not served."""
    return next((m.get("max_model_len") for m in served or [] if m["id"] == model), None)


EXPECTED_MAX_MODEL_LEN = 4096  # The --max-model-len value from 1D.4

after = show_models(new_url)
if context_length(after) is None:
    print(f"\n❌ {model} is not in this endpoint's model list. Check the URL and KLIQUE_MODEL.")
else:
    print(f"\nmax_model_len: {context_length(before)} -> {context_length(after)}")
    if context_length(after) == EXPECTED_MAX_MODEL_LEN:
        print("✅ The new limit is in effect.")
    else:
        print(f"❌ Expected {EXPECTED_MAX_MODEL_LEN}. This is the old instance, or --max-model-len was not applied.")

> **Checkpoint:** the cell prints `✅ The new limit is in effect.`, and the model name is unchanged.

## 1D.6 Test the Context Limit

vLLM checks a request's length before it generates anything. The first request below fits; the second repeats a sentence until the prompt alone is well past 4,096 tokens.

In [ ]:
def send(url, content, max_tokens):
    """Send one chat completion and print the outcome."""
    data = {
        "model": model,
        "messages": [{"role": "user", "content": content}],
        "max_tokens": max_tokens,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    response = requests.post(f"{url}/chat/completions", headers=headers, json=data, timeout=120)
    if response.ok:
        usage = response.json()["usage"]
        print(f"✅ HTTP 200: {usage['prompt_tokens']} prompt tokens + {usage['completion_tokens']} completion tokens")
    else:
        print(f"HTTP {response.status_code}: {response.text[:400]}")


short = "Summarize in one sentence: GPUs run many small calculations in parallel."
too_long = "Summarize this text. " + "GPUs run many small calculations in parallel. " * 700

send(new_url, short, max_tokens=64)
send(new_url, too_long, max_tokens=64)

The second request returns HTTP `400` with a message that it exceeds the model's maximum context length. The limit counts the prompt **plus** `max_tokens`, so a short prompt with a very large `max_tokens` is rejected the same way.

A client that sends long documents must check this error and shorten, split, or summarize its input. Without the argument, the same request would have been accepted and held GPU memory for its whole length.

## 1D.7 Watch Requests Queue

With `--max-num-seqs 4`, vLLM runs at most four requests at once. The cell below sends eight identical requests at the same time. None should fail; the second four wait for the first four to finish.

In [ ]:
MAX_TOKENS = 128
COUNT = 8


def timed_request(_):
    """Send one fixed-length request. Returns (seconds, HTTP status or error name)."""
    data = {
        "model": model,
        "messages": [{"role": "user", "content": "Write a short paragraph about GPUs."}],
        "max_tokens": MAX_TOKENS,
        "ignore_eos": True,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    start = time.perf_counter()
    try:
        response = requests.post(f"{new_url}/chat/completions", headers=headers, json=data, timeout=180)
    except requests.RequestException as error:
        return time.perf_counter() - start, type(error).__name__
    return time.perf_counter() - start, response.status_code


with ThreadPoolExecutor(max_workers=COUNT) as pool:
    results = sorted(pool.map(timed_request, range(COUNT)), key=lambda r: r[0])

for i, (seconds, status) in enumerate(results, start=1):
    print(f"{i}. {seconds:5.1f} s  {'' if status == 200 else f'❌ {status}'}")

Sorted by time, the latencies usually fall into two groups: four requests that ran right away, and four that took roughly twice as long because they waited for a slot. Compare this with the 8-concurrent row in Section 1B, where all eight ran together.

`ignore_eos` makes every request generate exactly `MAX_TOKENS` tokens, so the groups are easy to see. If the server rejects it, remove that line; the groups then blur because replies end at different lengths.

> **Checkpoint:** eight lines with no `❌`, and the slowest requests take noticeably longer than the fastest.

Neither limit is right or wrong. A low `--max-num-seqs` keeps each request fast and predictable but makes users wait in line under load; a high one serves more users at once, each a little slower. Your measurements from 1B are what you use to choose.

## 1D.8 Update `.env` and Later Sections

If the URL in 1D.5 differs from the one in `.env`, every later notebook still points at the stopped instance. Set `UPDATE_ENV = True` and run the cell to replace `KLIQUE_BASE_URL` in `.env`. The cell changes only that line; the model name and token stay as they are.

In [ ]:
from pathlib import Path

UPDATE_ENV = False  # Set to True to write the new URL into .env

env_path = Path("..") / ".env"
if new_url == base_url:
    print(".env already points at this endpoint. Nothing to change.")
elif not UPDATE_ENV:
    print(f"Not changed. Set UPDATE_ENV = True to write KLIQUE_BASE_URL={new_url}")
else:
    lines = env_path.read_text().splitlines()
    lines = [f"KLIQUE_BASE_URL={new_url}" if line.startswith("KLIQUE_BASE_URL=") else line for line in lines]
    env_path.write_text("\n".join(lines) + "\n")
    print(f"Updated KLIQUE_BASE_URL in {env_path.resolve()}. Restart the kernel of any open notebook to reload it.")

The model name did not change, because the new arguments do not include `--served-model-name`. `KLIQUE_MODEL` is still correct.

> **Note:** If you have already added the endpoint to an LLM Gateway profile in Section 2, that profile still refers to the original instance's endpoint. Update it to the new endpoint before continuing with Sections 2–4, or do this notebook after them. If you also set `--served-model-name`, update `KLIQUE_MODEL` in `.env` to the new name as well.

## 1D.9 Clean Up

You now have two instances: `qwen-lab` (stopped) and `qwen-lab-tuned` (running).

* Keep `qwen-lab-tuned` running if you are continuing to Section 2. The limits from this notebook do not affect the requests there.
* When you are finished, stop it as described in Section 1.9.
* You can **Delete** `qwen-lab` once you no longer need it. The exported configuration file lets you recreate it with the recipe's arguments.


## Troubleshooting

| Symptom | Likely cause |
|---|---|
| `qwen-lab-tuned` stays in **Loading** | The original instance is still holding the queue's GPU. Stop `qwen-lab` and check the console log. |
| Goes to **Stopped** with an argument error | A typo in the added arguments. Each one needs two dashes and a value, separated by spaces. |
| Goes to **Stopped** with an error about the KV cache or context length | `--max-model-len` is larger than the KV cache can hold on this GPU. Lower it. |
| `max_model_len` did not change | The notebook is still calling the original endpoint. Check the URL you entered in 1D.5. |
| The long request returns `200` | `--max-model-len` was not applied. Check the arguments on the instance and in its console log. |
| All eight requests finish at about the same time | `--max-num-seqs` was not applied, or `ignore_eos` was removed and the replies are short. |
| `404` from every request after updating `.env` | The URL lacks `/v1` or points at a stopped instance. Compare it with Model Endpoints. |

## Next: Putting the Endpoint Behind the LLM Gateway

You can now deploy a model, measure it, constrain its output, and tune how vLLM serves it. Every client still needs this endpoint's URL, its exact model name, and a token, and nothing governs who uses it or how much.

**Next:** In Section 2, [Use the LLM Gateway](../02_LLMGateway/02a_Use_the_LLM_Gateway.ipynb), we put the endpoint behind the LLM Gateway and call it the governed way.